In [1]:
import os
import re
import html
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

/home/tommaso/projects/Board-Game-Data-Analytics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Caching: natural-language embeddings & FAISS index

The dense-retrieval pipeline uses the same natural-language description source as the BM25 notebook.
The original BGG description is recovered from `details.csv`, aligned by `BGGId`, cleaned only for
HTML entities and whitespace, and stored in `Description_nat`.

- **`data/embeddings_nat.npy`** - embedding matrix generated from `Description_nat`, row-aligned with `df`.
- **`data/faiss_index.bin`** - the FAISS `IndexHNSWFlat` built from those embeddings.

`df` is rebuilt from the source CSVs on each run. The embedding matrix and FAISS index are cached to
disk because encoding and index construction are the expensive steps. Both caches are validated against
`len(df)` when loaded.

Delete `embeddings_nat.npy` and/or `faiss_index.bin` to force a rebuild.


In [2]:
DATA_DIR = '../data'
DETAILS_PATH = os.path.join(DATA_DIR, 'details.csv')
EMBEDDINGS_PATH = os.path.join(DATA_DIR, 'embeddings_nat.npy')
FAISS_INDEX_PATH = os.path.join(DATA_DIR, 'faiss_index.bin')

# 1. Rebuild the base dataframe from the source CSVs.
df_clean = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_dataset.csv'))
df_raw = pd.read_csv(os.path.join(DATA_DIR, 'DM1_game_dataset.csv'))
df_text = df_raw[['BGGId', 'Description']]
df = pd.merge(df_clean, df_text, on='BGGId', how='inner')
df = df.dropna(subset=['Description']).reset_index(drop=True)

# 2. Recover the original natural-language description from details.csv.
#    BGGId is the stable identifier used to align the two datasets.
orig = pd.read_csv(DETAILS_PATH).drop_duplicates('id')
m = df[['BGGId']].merge(orig[['id', 'description']], left_on='BGGId', right_on='id', how='left')
assert len(m) == len(df), 'merge changed the number of rows'

def clean_natural_text(text):
    if not isinstance(text, str):
        return ''
    text = html.unescape(html.unescape(text))
    return re.sub(r'\s+', ' ', text).strip()

natural_text = m['description'].map(clean_natural_text)
has_natural = natural_text.str.len() > 0
df['Description_nat'] = natural_text.where(has_natural, df['Description']).values

print(f'Dataset ready! Total board games: {len(df)}')
print(f'Games using the original natural-language description: {has_natural.sum()} / {len(df)}')
print(f'Games falling back to DM1 description: {(~has_natural).sum()} / {len(df)}')


Dataset ready! Total board games: 19728
Games using the original natural-language description: 19256 / 19728
Games falling back to DM1 description: 472 / 19728


## Generate dense embeddings from natural-language descriptions


**Model: `all-MiniLM-L6-v2`.** The same lightweight sentence-transformer is used for the dense-retrieval
baseline. It encodes `Description_nat` directly, preserving the natural-language wording rather than
using the BM25-tokenized representation. The 384-dimensional vectors provide a compact representation
for semantic search and keep the FAISS index small and fast to query.


In [3]:
# Load the pre-trained Neural IR model
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = None
if os.path.exists(EMBEDDINGS_PATH):
    cached_embeddings = np.load(EMBEDDINGS_PATH)
    if cached_embeddings.shape[0] == len(df):
        embeddings = cached_embeddings.astype('float32')
        print(f'Natural-language embeddings loaded from cache! Shape: {embeddings.shape}')
    else:
        print(
            f"Cache found at '{EMBEDDINGS_PATH}' but row count doesn't match df "
            f"({cached_embeddings.shape[0]} vs {len(df)}) - treating as stale, will recompute."
        )

if embeddings is None:
    descriptions = df['Description_nat'].tolist()

    print('Encoding natural-language descriptions into dense vectors...')
    embeddings = model.encode(descriptions, show_progress_bar=True)
    embeddings = np.asarray(embeddings).astype('float32')

    print(f'Natural-language embeddings generated! Shape: {embeddings.shape}')

    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"Cached natural-language embeddings to '{EMBEDDINGS_PATH}'.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16465.18it/s]


Natural-language embeddings loaded from cache! Shape: (19728, 384)


## Approximate k-NN with FAISS (HNSW)

The FAISS index is built from the embeddings generated from `Description_nat`. If a cached
`faiss_index.bin` exists and its vector count matches `len(df)`, it is loaded directly; otherwise
the HNSW graph is rebuilt from the current natural-language embeddings and persisted for reuse.

Keeping the embedding cache and the FAISS index as separate artifacts makes the row-count consistency
check explicit and allows either artifact to be regenerated independently.


In [4]:
# Get the vector dimension size (384)
dimension = embeddings.shape[1]

index = None
if os.path.exists(FAISS_INDEX_PATH):
    cached_index = faiss.read_index(FAISS_INDEX_PATH)
    if cached_index.ntotal == len(df):
        index = cached_index
        print(f"FAISS index loaded from cache! Total vectors: {index.ntotal}")
    else:
        print(
            f"Cache found at '{FAISS_INDEX_PATH}' but vector count doesn't match df "
            f"({cached_index.ntotal} vs {len(df)}) - treating as stale, will rebuild."
        )

if index is None:
    # Initialize the HNSW index
    # The '32' is the number of connections each node has in the graph
    index = faiss.IndexHNSWFlat(dimension, 32)

    print("Building the HNSW graph...")
    index.add(embeddings)
    print(f"Index built! Total vectors in FAISS: {index.ntotal}")

    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"Cached FAISS index to '{FAISS_INDEX_PATH}'.")

FAISS index loaded from cache! Total vectors: 19728


## Semantic Query Processing

In [5]:
def search_games_faiss(query, top_k=5):
    """Search the FAISS index using semantic embeddings."""
    # 1. Convert the user query into a dense vector.
    query_vector = model.encode([query]).astype('float32')

    # 2. Perform the approximate k-NN search (L2 distance).
    distances, indices = index.search(query_vector, top_k)

    # 3. Retrieve the matching games.
    results = df.iloc[indices[0]].copy()
    results['faiss_distance'] = distances[0]

    return results[['Name', 'faiss_distance', 'Description_nat']]

### Test the neural engine

The same natural-language query used in NB6 is used here so that BM25 and dense retrieval can be compared on identical input.


In [6]:
user_query = "A cooperative space game involving betrayal"
results = search_games_faiss(user_query, top_k=5)

print(f"Semantic Results for: '{user_query}'\n")
for i, row in results.iterrows():
    print(f"{row['Name']} (Distance: {row['faiss_distance']:.2f})")
    print(f"Snippet: {row['Description_nat'][:150]}...\n")


Semantic Results for: 'A cooperative space game involving betrayal'

Cosmic Encounter (Distance: 0.76)
Snippet: Build a galactic empire... In the depths of space, the alien races of the Cosmos vie with each other for control of the universe. Alliances form and s...

Captain Future (Distance: 0.87)
Snippet: A Mysterious Universe. Silver and glass rockets jet around the galaxy. Other planets are now being populated by courageous scientists and spacemen. Ca...

The Crew: The Quest for Planet Nine (Distance: 0.87)
Snippet: In the co-operative trick-taking game The Crew: The Quest for Planet Nine, the players set out as astronauts on an uncertain space adventure. What are...

2491 Planetship (Distance: 0.90)
Snippet: Based on City of Spies: Estoril 1942, 2491 Planetship reimplements the mechanisms from a city of spies to a new fictional universe filled with rich lo...

Nowhere to Go (Distance: 0.92)
Snippet: Two-player strategy game with a spy theme on a unique hexagonal game board with spa